# Senate Amendments Considered

Amendments agreed to or not agreed to on the Senate floor, dated by latest action. See the README for definitions.

`fetch_amendments` uses the Congress.gov API and runs headless. It needs a free key from https://api.congress.gov/sign-up/, saved in a file named `.congress_api_key` at the top of the repo (or in the `CONGRESS_API_KEY` environment variable). The first run for a Congress takes about 45 minutes; responses are cached in `.api_cache`, so later runs are quick.

(`scrape_amendments` is the original congress.gov scraper. It gives the same columns but opens a visible Chrome window.)

In [ ]:
import pandas as pd
import amendments_helper

START, END = "01/01/2025", None   # MM/DD/YYYY, inclusive; END=None means today
MAJORITY = "R"                    # party controlling the Senate: D for the 117th-118th, R for the 119th

In [ ]:
df = amendments_helper.fetch_amendments(START, END)
outfile = f"amendments_{pd.Timestamp(START):%Y-%m-%d}_to_{pd.Timestamp(END or 'today'):%Y-%m-%d}.csv"
df.to_csv(outfile, index=False)
print(len(df), "amendments saved to", outfile)
df.head()

## Rows to check by hand
The classification reads the latest action's text. Rows it cannot classify (for example, "Senate vitiated previous adoption of amendment...") are marked INPUT BY HAND.

Note: the Outcome code for "not agreed to" is `NA`. When reading these CSVs back into pandas, use `keep_default_na=False`, or pandas will turn `NA` into a missing value.

In [ ]:
df[(df["Outcome"] == "INPUT BY HAND") | (df["Action Type"] == "INPUT BY HAND")]

## Quarterly counts for the full data tables
Per-quarter counts (not cumulative). The published sheet is cumulative within each Congress, so add each quarter to the previous one. Independents are counted with the party they caucus with (Democrats). The four "amended" categories are mutually exclusive, as in the methodology: amendments to amendments, then budget resolutions (Con), then other resolutions (Res), and everything else is a bill (treaties included). The Budget Resolutions row is published as-is (not cumulative).

In [ ]:
amendments_helper.quarterly_summary(df, majority_party=MAJORITY)